# Exploratory Data Analysis — Customer Churn

**Methodology:** CRISP-DM Phase 2 — Data Understanding

This notebook covers:
1. Dataset overview and schema inspection
2. Univariate analysis (distributions)
3. Bivariate analysis (feature vs. churn)
4. Correlation matrix
5. Insights for feature engineering

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.generator import SyntheticDataGenerator
from src.data.validator import DataValidator
from src.visualization.plots import ChurnPlotter

pd.options.display.float_format = '{:.3f}'.format
plotter = ChurnPlotter()

## 1. Load Data

In [ ]:
df = SyntheticDataGenerator(random_state=42).generate(n_samples=10_000)
print(f'Shape: {df.shape}')
df.head()

## 2. Data Quality Check

In [ ]:
report = DataValidator().validate(df)
print(f'Validation passed: {report.passed}')
print(f'Missing values: {report.missing_values}')

df.describe(include='all').T

## 3. Target Distribution

In [ ]:
plotter.churn_distribution(df['churned']).show()

## 4. Churn Rate by Segment

In [ ]:
for col in ['contract_type', 'payment_method', 'internet_service', 'segment']:
    plotter.churn_by_segment(df, col).show()

## 5. Numeric Feature Distributions

In [ ]:
numeric_cols = ['tenure_months', 'monthly_charges', 'total_charges',
                'num_products', 'num_support_calls']

fig = make_subplots(rows=2, cols=3, subplot_titles=numeric_cols)
for i, col in enumerate(numeric_cols):
    row, c = divmod(i, 3)
    for label, color in [(0, '#22C55E'), (1, '#EF4444')]:
        fig.add_trace(
            go.Histogram(
                x=df[df['churned']==label][col],
                name='retained' if label==0 else 'churned',
                marker_color=color, opacity=0.65, showlegend=(i==0)
            ),
            row=row+1, col=c+1
        )
fig.update_layout(barmode='overlay', height=500, title='Numeric Features by Churn')
fig.show()

## 6. Correlation Matrix

In [ ]:
corr = df[numeric_cols + ['churned']].corr()
fig = px.imshow(
    corr, text_auto='.2f', color_continuous_scale='RdBu_r',
    title='Correlation Matrix', aspect='auto'
)
fig.show()

print('\nCorrelation with target (churned):')
print(corr['churned'].drop('churned').sort_values(ascending=False))

## 7. Key Insights

| Finding | Implication |
|---|---|
| `contract_type=month-to-month` has 3× higher churn rate | Strong categorical predictor; consider interaction with tenure |
| `tenure_months` is negatively correlated with churn | Loyal customers rarely leave — add logarithmic encoding |
| `num_support_calls` correlates positively with churn | Proxy for dissatisfaction; consider rate normalisation |
| `electronic_check` payment method → higher churn | May indicate lower commitment / automatic payment barriers |
| `is_senior_citizen=True` → slightly elevated churn | Could indicate digital literacy barrier with Fiber service |